In [ ]:
import os, sys, json, hashlib, shutil, zipfile, numpy as np, glob
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

# Michael Rustad source code
SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

print('Environment ready')
print('SRC exists:', SRC.exists())


In [ ]:
# Load busyaprime factorization
import shutil
bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(bp_members / 'dfthcsos__submission.npz', sub_dir / 'dfthcsos' / 'submission.npz')

fac = dict(np.load(sub_dir / 'dfthcsos' / 'submission.npz', allow_pickle=False))
print('Factorization keys:', list(fac.keys()))
for k in fac:
    print(f'  {k}: shape={fac[k].shape}, dtype={fac[k].dtype}')


In [ ]:
# Run the optimization pipeline
import subprocess, time
os.environ['QCC_ROOT'] = str(WORK)

IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
BUILDS = WORK / 'builds'
BUILDS.mkdir(exist_ok=True)

TAG = 'bp_opt'
CAP = '0.0193'

# Step 1: finish.py (one-body refit)
print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / f'{TAG}_fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Step 2: gauge.py (Gram-preserving column rotations)
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / f'{TAG}_fin.npz'), str(RUNS / f'{TAG}_g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Step 3: lamopt.py (lambda_eff descent)
print('Step 3: lamopt.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'lamopt.py'), str(RUNS / f'{TAG}_g.npz'), str(RUNS / f'{TAG}_lam.npz'),
     '--steps', '1500', '--lr-w', '5e-7', '--lr-o', '2e-6', '--log', '500', '--cap', CAP],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print(result.stdout[-1000:] if result.stdout else 'NO STDOUT')
print(result.stderr[-1000:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Step 4: finish.py again
print('Step 4: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(RUNS / f'{TAG}_lam.npz'), str(RUNS / f'{TAG}_lamfin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Step 5: gauge.py again
print('Step 5: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / f'{TAG}_lamfin.npz'), str(RUNS / f'{TAG}_final.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Step 6: build + verify
print('Step 6: build_dft_local.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'build_dft_local.py'), str(RUNS / f'{TAG}_final.npz'), str(BUILDS / f'dft_{TAG}')],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers'))
)
print(result.stdout[-1000:] if result.stdout else 'NO STDOUT')
print(result.stderr[-1000:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')


In [ ]:
# Verify with official verifier
from dfthcsos.verify import verify as CV
r = CV(BUILDS / f'dft_{TAG}' / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

# Package
zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
